In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
# import rasterio
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn


2026-02-08 13:57:53.158152: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only GPU 1

In [3]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    @tf.function
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    @tf.function
    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 7]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 7]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [7, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [7, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

# filenames = ['/content/drive/MyDrive/GeoSR/L8MODIS_30000_2/L8MODIS.tfrecords']
# filenames = ['/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_0.tfrecords']
# ds = input_pipeline(filenames, batch_size=5, is_shuffle=False, is_train=True, is_repeat=True)

# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))*0.0001
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))*0.0000275-0.2

#         axes[0].imshow(lres_img[:, :, [0, 3, 2]]*2)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, 3:0:-1]*2)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

    # break


# SeD

In [4]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/SeD_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/SeD_M2L8


/glade/derecho/scratch/lizhili/SeD/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
from models import sed, model_rrdb
import yaml

In [6]:
opt_path = './options/train_rrdb_P+SeD.yml'
with open(opt_path, 'r') as f:
    opt = yaml.safe_load(f)

In [7]:
model_ex = sed.CLIP_Semantic_extractor(**opt['model_ex']).to('cuda')
model = model_rrdb.RRDBNet(**{'in_nc': 7, 'out_nc': 7}).to('cuda')
model_d = sed.SeD_P(**opt['model_d']).to('cuda')

In [ ]:
def load_matched_weights(model, checkpoint_path):
    # Load checkpoint (state_dict)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    # print(state_dict.keys())
    # state_dict = state_dict['params']

    # Filter only matching keys
    model_dict = model.state_dict()
    # print(model_dict.keys())
    matched_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

    # Load the matched weights
    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Loaded {len(matched_dict)} matching parameters out of {len(model_dict)} total.")

    return model

model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/SED_G_x4_epoch_weights_finetune_M2L8_new.pth')
model_d = load_matched_weights(model_d, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/SED_D_x4_epoch_weights_finetune_M2L8_new.pth')
# model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/SED_G_x16_epoch_weights_finetune_M2L8_new.pth')
# model_d = load_matched_weights(model_d, '/glade/derecho/scratch/lizhili/m2l8/SED_D_x16_epoch_weights_finetune_M2L8_new.pth')

In [9]:
model_fixed = model_rrdb.RRDBNet(**opt['model']['rrdb']).to('cuda')
model_fixed = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/SED_G_x4_epoch_weights_finetune_M2L8_new.pth')

✅ Loaded 702 matching parameters out of 702 total.


In [10]:
import torch.optim as optim
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], **opt['optimizer'])
optimizer_d = optim.Adam([p for p in model_d.parameters() if p.requires_grad], **opt['optimizer_d'])
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, **opt['scheduler'])
scheduler_d = optim.lr_scheduler.MultiStepLR(optimizer_d, **opt['scheduler_d'])

In [11]:
from utils import losses
loss_weight = opt['loss_weights']
loss_pix = torch.nn.L1Loss()
loss_pix = loss_pix.to('cuda')
loss_g = losses.GANLoss(gan_type='vanilla', loss_weight=loss_weight['loss_g']).to('cuda')
loss_dict_per = {'2': 0.1, '7': 0.1, '16': 1.0, '25': 1.0, '34': 1.0}
loss_p = losses.PerceptualLoss(layer_weights=loss_dict_per, perceptual_weight=loss_weight['loss_p'], criterion='l1').to('cuda')

/glade/derecho/scratch/lizhili/SeD/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/glade/derecho/scratch/lizhili/SeD/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [12]:
def random_crop_lr_hr(img_lr, img_hr, lr_crop_size):
    """
    Random aligned crop for super-resolution pairs.

    img_lr: Tensor [B, C, 64, 64]
    img_hr: Tensor [B, C, 1024, 1024]
    lr_crop_size: int (e.g., 32)

    Returns:
        lr_crop: [B, C, lr_crop_size, lr_crop_size]
        hr_crop: [B, C, lr_crop_size*16, lr_crop_size*16]
    """
    scale = img_hr.shape[-1] // img_lr.shape[-1]  # 16

    _, _, H_lr, W_lr = img_lr.shape
    assert H_lr >= lr_crop_size and W_lr >= lr_crop_size

    top_lr = torch.randint(0, H_lr - lr_crop_size + 1, (1,)).item()
    left_lr = torch.randint(0, W_lr - lr_crop_size + 1, (1,)).item()

    top_hr = top_lr * scale
    left_hr = left_lr * scale

    lr_crop = img_lr[:, :, top_lr:top_lr+lr_crop_size,
                             left_lr:left_lr+lr_crop_size]

    hr_crop = img_hr[:, :, top_hr:top_hr+lr_crop_size*scale,
                             left_hr:left_hr+lr_crop_size*scale]

    return lr_crop, hr_crop

In [ ]:
from collections import OrderedDict

# Training starts
total_epochs = 30

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']
ds = input_pipeline(filenames, batch_size=6, is_shuffle=True, is_train=True, is_repeat=False)


    
for epoch in range(total_epochs):
    print(f'Epoch {epoch}')
    for step, (lr, hr) in enumerate(ds):

        learning_r = optimizer.param_groups[0]['lr']

        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2

        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
        
        if step == 0:
            print(lr.shape)
            print(hr.shape)

        hr_semantic = model_ex(hr[:, :3, :, :])

        loss_dict = OrderedDict()

        with torch.no_grad():
            features = model_fixed(lr)
            features = torch.clamp(features, 0, 1)

        sr = model(features)

        for p in model_d.parameters():
            p.requires_grad = False

        optimizer.zero_grad()

        l_g_total = 0

        # pixel loss
        loss_pixel = loss_pix(sr, hr)
        l_g_total += loss_pixel * loss_weight['loss_pix']
        loss_dict['loss_pix'] = loss_pixel.item()
        # perceptual loss
        loss_percep = loss_p(sr[:, :3, :, :], hr[:, :3, :, :])
        l_g_total += loss_percep
        loss_dict['loss_p'] = loss_percep.item()
        # gan loss
        fake_g_pred = model_d(sr, hr_semantic)
        loss_gan = loss_g(fake_g_pred, True, is_disc=False)
        l_g_total += loss_gan
        loss_dict['loss_g'] = loss_gan.item()
        l_g_total.backward()
        optimizer.step()
        scheduler.step()

        # optimize net_d
        for p in model_d.parameters():
            p.requires_grad = True

        optimizer_d.zero_grad()
        # real
        real_d_pred = model_d(hr, hr_semantic)
        l_d_real = loss_g(real_d_pred, True, is_disc=True)
        loss_dict['l_d_real'] = l_d_real.item()
        l_d_real.backward()
        # fake
        fake_d_pred = model_d(sr.detach().clone(), hr_semantic)  # clone for pt1.9
        l_d_fake = loss_g(fake_d_pred, False, is_disc=True)
        loss_dict['l_d_fake'] = l_d_fake.item()
        l_d_fake.backward()
        optimizer_d.step()
        scheduler_d.step()

        if step % 500 == 0:
            print(f'Step {step}, Loss: {loss_pixel.item()}')
            # torch.save(model, '/content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/SED_4x.pt')
            # torch.save(model_d, '/content/drive/MyDrive/NAIP_SuperRes_DS/NLCD_SR/SED_D_4x.pt')

    torch.save(model.state_dict(), '/glade/derecho/scratch/lizhili/m2l8/SED_G_x16_epoch_weights_finetune_M2L8_new.pth')
    torch.save(model_d.state_dict(), '/glade/derecho/scratch/lizhili/m2l8/SED_D_x16_epoch_weights_finetune_M2L8_new.pth')

In [ ]:
from collections import OrderedDict


def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_gmodel,
                    finetuned_dmodel,
                    output_tfrecords,
                    model, model_d, model_fix):

    num_test = num_sample-num_training
    model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/SED_G_x16_epoch_weights_finetune_M2L8_new.pth')
    model_d = load_matched_weights(model_d, '/glade/derecho/scratch/lizhili/m2l8/SED_D_x16_epoch_weights_finetune_M2L8_new.pth')
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], **opt['optimizer'])
    optimizer_d = optim.Adam([p for p in model_d.parameters() if p.requires_grad], **opt['optimizer_d'])


    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch


    #--------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 8, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')
        scaler = torch.cuda.amp.GradScaler()

        for step, (lr, hr, _) in enumerate(ds):
            learning_r = optimizer.param_groups[0]['lr']

            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
            lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

            lr, hr = random_crop_lr_hr(lr, hr, lr_crop_size=32)
            if step == 0:
                print(lr.shape)
                print(hr.shape)

            hr_semantic = model_ex(hr[:, :3, :, :])

            loss_dict = OrderedDict()

            with torch.no_grad():
                features = model_fix(lr)
                features = torch.clamp(features, 0, 1)

            sr = model(features)

            for p in model_d.parameters():
                p.requires_grad = False

            optimizer.zero_grad()

            l_g_total = 0

            # pixel loss
            loss_pixel = loss_pix(sr, hr)
            l_g_total += loss_pixel * loss_weight['loss_pix']
            loss_dict['loss_pix'] = loss_pixel.item()
            # perceptual loss
            loss_percep = loss_p(sr[:, :3, :, :], hr[:, :3, :, :])
            l_g_total += loss_percep
            loss_dict['loss_p'] = loss_percep.item()
            # gan loss
            fake_g_pred = model_d(sr, hr_semantic)
            loss_gan = loss_g(fake_g_pred, True, is_disc=False)
            l_g_total += loss_gan
            loss_dict['loss_g'] = loss_gan.item()
            l_g_total.backward()
            optimizer.step()
            scheduler.step()

            # optimize net_d
            for p in model_d.parameters():
                p.requires_grad = True

            optimizer_d.zero_grad()
            # real
            real_d_pred = model_d(hr, hr_semantic)
            l_d_real = loss_g(real_d_pred, True, is_disc=True)
            loss_dict['l_d_real'] = l_d_real.item()
            l_d_real.backward()
            # fake
            fake_d_pred = model_d(sr.detach().clone(), hr_semantic)  # clone for pt1.9
            l_d_fake = loss_g(fake_d_pred, False, is_disc=True)
            loss_dict['l_d_fake'] = l_d_fake.item()
            l_d_fake.backward()
            optimizer_d.step()
            scheduler_d.step()

            if step % 50 == 0:
                print(f'Step {step}, Loss: {loss_pixel.item()}')

        torch.save(model.state_dict(), finetuned_gmodel)
        torch.save(model_d.state_dict(), finetuned_dmodel)

    #--------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 2, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)

        with torch.no_grad():
            features = model_fixed(lr)
            features = torch.clamp(features, 0, 1)
            output = model(features)
            
        output = output.detach().cpu().numpy()
        # output = np.clip(output, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        output[output<0]=0
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(2*lr_show[:, :, [0,3,2]])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(2*(output_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_gmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_g_x16_M2L8_River_Finetune.pth'
finetuned_dmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_d_x16_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_SED_x16.tfrecords'
model_fix = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/x4/SeD_x4_downstream_models/SED_g_x4_M2L8_River_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_d, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_gmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_g_x16_M2L8_Urban_Finetune.pth'
finetuned_dmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_d_x16_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_SED_x16.tfrecords'
model_fix = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/x4/SeD_x4_downstream_models/SED_g_x4_M2L8_Urban_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_d, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_gmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_g_x16_M2L8_CDL_Finetune.pth'
finetuned_dmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_d_x16_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_SED_x16.tfrecords'
model_fix = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/x4/SeD_x4_downstream_models/SED_g_x4_M2L8_CDL_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_d, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_gmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_g_x16_M2L8_GPP_Finetune.pth'
finetuned_dmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_d_x16_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_SED_x16.tfrecords'
model_fix = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/x4/SeD_x4_downstream_models/SED_g_x4_M2L8_GPP_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_d, model_fix)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_gmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_g_x16_M2L8_CHM_Finetune.pth'
finetuned_dmodel = '/glade/derecho/scratch/lizhili/m2l8/SED_d_x16_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_SED_x16.tfrecords'
model_fix = load_matched_weights(model_fixed, '/glade/derecho/scratch/lizhili/m2l8/x4/SeD_x4_downstream_models/SED_g_x4_M2L8_CHM_Finetune.pth')

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_gmodel,
                finetuned_dmodel,
                output_tfrecords,
                model, model_d, model_fix)